# SwarmSort paper: real-world sample images and per-image statistics (Gen 2 models)

**What:** runs `paper/figures/scripts/r_export_realworld.py` on the 468 real-world test photos (TACO + HITL) of the
Gen 2 build. Each Gen 2 model (A: defaults, B: random search, C: PSO; `app/models/gen2/arm_{a,b,c}.pt` from the
clone) predicts every photo with the settings of the sample figure (confidence 0.25, NMS IoU 0.7, imgsz 640, CPU).
Its boxes are matched to the ground truth (greedy by confidence, same class, IoU >= 0.5), giving TP, FP and FN per
image and per model. An image is *fully correct* for a model when every ground-truth box is found with the right
class and there is no extra box. The script then picks three fixed selections, never by eye:

- **A, random rows:** the rule of `r_samples.py` (images with >= 3 boxes, seed-0 shuffle, first 2). This is the
  missing real-world row of the sample figure.
- **B, best cases** (deliberately favorable): images with >= 3 boxes that all three models get fully correct; if
  fewer than 3 qualify, the rule relaxes to >= 2 boxes, then to at least 2 of the 3 models. Ranked by the number of
  boxes, ties in the seed-0 order; the first 3. The level used and each model's totals over all 468 photos (how many
  it gets fully correct, mean per-image F1) are recorded, so the caption can say how favorable the choice is.
- **C, single-object successes** (deliberately favorable): images with exactly 1 box that all three models get fully
  correct, put in the seed-0 order as in A; the first 2. In the best-case figure they are drawn below the B images.
  (In the first run B found only 2 images, both with 2 boxes: almost no image with 2 or more boxes is fully correct
  for all three models.)

The totals also give the fully correct counts **by objects per image** (1, 2, >= 3), for each model and for all
three, so the caption can say how common each kind of success is.

**Why:** the paper shows where the detector fails (India). These are the counterpart: where it works. The
real-world test photos exist only here on Kaggle, inside the dataset built by `g2_0_build_dataset`, so the selection
is made here and a small bundle is downloaded; the figures are drawn locally by `r_samples.py`.

**Before you run it:** push the latest commits. The clone needs `paper/figures/scripts/r_export_realworld.py` and
`r_samples.py` (the notebook stops with a message if they are missing).

**Input** (Add Input -> Notebook Output): `g2_0_build_dataset`, its finished version (the test images,
`gen2_data/`). Nothing else: the Gen 2 weights and the recorded results come with the clone, and every weights file
is checked against the sha256 recorded when g2_6 scored it (`results/gen2/test_all_results.json`).

**Accelerator: none needed** (CPU is fine; the predictions run on the CPU anyway). **Internet: on** (for the clone).
Run it with **Save Version -> Save & Run All (Commit)**.

**Expected time:** about 10-15 minutes (copying the build to local disk, then 3 x 468 predictions on the CPU).

**Output:** `/kaggle/working/realworld_samples.zip` (a few hundred kB: the selected images and their labels,
`selection.json`, `per_image_stats.json`), and the same files unzipped in `/kaggle/working/realworld_samples/`.

**Freeze protection** (as in g2_7). The attached dataset is first copied to local disk, and the export runs under a
watchdog that restarts it after 20 minutes without output or file changes (at most 3 times).

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
assert os.path.isdir(f"{REPO}/src"), "git clone failed: turn on Settings -> Internet and run again"
%cd {REPO}
# Kaggle's own torch build is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| predictions run on the CPU")
EXPORT = "paper/figures/scripts/r_export_realworld.py"
for f in ("src/watchdog_run.py", EXPORT, "paper/figures/scripts/r_samples.py"):
    if not os.path.isfile(f):
        raise RuntimeError(f"{f} is not on GitHub yet: push the latest commits")
sys.path.insert(0, f"{REPO}/src")
from watchdog_run import run_watched
# About 10 s: check on this machine that a frozen script and its worker process are killed and retried
subprocess.run([sys.executable, "-m", "unittest", "-q",
                "test_watchdog.TestWatchdog.test_b_stalled_script_is_killed_and_retried",
                "test_watchdog.TestWatchdog.test_e_child_processes_are_killed_too"], cwd="tests", check=True)


def run(*args, watch=(), stall_minutes=20, max_retries=3):
    """Run a repository script under the stall watchdog (src/watchdog_run.py). Its output is streamed here. If it
    prints nothing and nothing changes under `watch` for stall_minutes, it is killed and started again with the same
    arguments, at most max_retries times. If the script stops with an error, this cell fails and the version is
    marked failed (as with check=True)."""
    args = [str(a) for a in args]
    print("$ python", " ".join(args), flush=True)
    run_watched([sys.executable, *args], watch=watch, stall_minutes=stall_minutes, max_retries=max_retries,
                env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
# Find the dataset built by g2_0_build_dataset (attached as input), copy it to this machine's local disk, and write
# configs that point at the local copy: /kaggle/working/gen2_cfg/data.yaml, data_test_<domain>.yaml, split.csv, lists/
# Why the copy: two Gen 2 runs froze when the attached notebook-output mount under /kaggle/input stopped answering.
# The export reads only the local copy. It goes outside /kaggle/working, so it does not become part of the output.
# The copy reads that mount too, so it fails loudly (TimeoutError) if no file arrives for 2 minutes.
import glob, json, shutil
from watchdog_run import copy_files, list_tree
INPUT, CFG = "/kaggle/input", "/kaggle/working/gen2_cfg"
builds = sorted({os.path.dirname(p) for p in glob.glob(f"{INPUT}/**/build_report.json", recursive=True)
                 if os.path.isfile(os.path.join(os.path.dirname(p), "data.yaml"))
                 and os.path.isdir(os.path.join(os.path.dirname(p), "images"))})
if not builds:
    raise FileNotFoundError("The Generation 2 dataset is not attached. Add Input -> Notebook Output -> "
                            "g2_0_build_dataset (its finished version), then run again.")
if len(builds) > 1:
    raise RuntimeError("Several built datasets are attached; remove all but one version of g2_0_build_dataset:\n  "
                       + "\n  ".join(builds))
BUILD_SRC = builds[0]
print("Built dataset (attached):", BUILD_SRC)
files = list_tree(BUILD_SRC, exclude=["*.cache"])
size = sum(n for _, n in files)
for base in ("/kaggle/tmp", "/kaggle/temp", "/tmp"):  # local disk outside /kaggle/working
    if os.path.isdir(base) and os.access(base, os.W_OK) and shutil.disk_usage(base).free > 1.1 * size + 2e9:
        BUILD = f"{base}/gen2_data"
        break
else:
    raise OSError(f"No local folder has {size / 1e9:.1f} GB free for the dataset copy")
stamp = {"source": BUILD_SRC, "files": len(files), "bytes": size}
if os.path.isfile(f"{BUILD}/.copied.json") and json.load(open(f"{BUILD}/.copied.json")) == stamp:
    print("Local copy already made in this session:", BUILD)
else:
    shutil.rmtree(BUILD, ignore_errors=True)
    print(f"Copying {len(files)} files ({size / 1e9:.2f} GB) to {BUILD}", flush=True)
    info = copy_files(BUILD_SRC, BUILD, files, workers=16, stall_seconds=120)
    if list_tree(BUILD) != files:
        raise RuntimeError(f"The local copy {BUILD} differs from {BUILD_SRC} (file names or sizes)")
    json.dump(stamp, open(f"{BUILD}/.copied.json", "w"))
    print(f"Copied {info['files']} files, {info['bytes'] / 1e9:.2f} GB in {info['seconds']:.0f} s "
          f"({info['bytes'] / 1e6 / max(info['seconds'], 1):.0f} MB/s); names and sizes match the attached build")
for sub in ("images", "labels"):
    print(f"  {sub}/: {len(os.listdir(f'{BUILD}/{sub}'))} files")
if "--relocate-from" not in open("src/build_dataset.py", encoding="utf-8").read():
    raise RuntimeError("src/build_dataset.py on GitHub has no --relocate-from mode yet: push the latest commits")
run("src/build_dataset.py", "--relocate-from", BUILD, "--out", CFG)
DATA = f"{CFG}/data_test_real_world.yaml"
if not os.path.isfile(DATA):
    raise FileNotFoundError(f"{DATA} missing: the attached build has no real_world test set (see g2_0_build_dataset)")
if not os.path.isfile(f"{CFG}/split.csv"):
    raise FileNotFoundError(f"{CFG}/split.csv missing: the relocation did not write it")
print(open(DATA).read())


In [ ]:
# The three Gen 2 models from the clone (app/models/gen2/arm_<x>.pt). Each file's sha256 must equal the sha256_12
# recorded when g2_6 scored that model on the test set (results/gen2/test_all_results.json). The expected size of the
# real-world test set comes from the g2_6 result of that set (results/gen2/test_real_world_results.json).
import hashlib
RECORDED, REAL = "results/gen2/test_all_results.json", "results/gen2/test_real_world_results.json"
for f in (RECORDED, REAL):
    if not os.path.isfile(f):
        raise FileNotFoundError(f"{f} is not in the clone: commit it and push")
recorded = {label: m["sha256_12"] for label, m in json.load(open(RECORDED))["models"].items()}
real = json.load(open(REAL))
N_IMAGES, N_BOXES = real["images"], real["boxes"]
print(f"Real-world test set scored in g2_6: {N_IMAGES} images, {N_BOXES} boxes")


def sha12(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:12]


WEIGHTS = []
for x in "abc":
    w = f"app/models/gen2/arm_{x}.pt"
    if not os.path.isfile(w):
        raise FileNotFoundError(f"{w} is not in the clone: commit app/models/gen2/ and push")
    got, want = sha12(w), recorded[x.upper()]
    if got != want:
        raise RuntimeError(f"{w} has sha256 {got}, but arm {x.upper()} was scored with {want} ({RECORDED})")
    print(f"  arm {x.upper()}: {w}  sha256 {got}  (matches {RECORDED})")
    WEIGHTS.append(w)


In [ ]:
# The export: predictions, matching and per-image statistics on all real-world test images, the two fixed
# selections, the selected images and labels, and the zip. --expect stops it unless the test set has exactly the
# images and boxes that g2_6 scored.
OUT = "/kaggle/working/realworld_samples"
ZIP = "/kaggle/working/realworld_samples.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
run(EXPORT, "--data", DATA, "--weights", *WEIGHTS, "--out", OUT, "--expect", N_IMAGES, N_BOXES, watch=[OUT])
for f in (ZIP, f"{OUT}/selection.json", f"{OUT}/per_image_stats.json"):
    if not os.path.isfile(f):
        raise FileNotFoundError(f"the export did not write {f}")


In [ ]:
# Summary of the bundle (also in selection.json): totals per model, the fully correct counts by objects per image,
# the rule levels of the best cases, the selected images (A, B, C), and the zip's contents and size
import zipfile
sel = json.load(open(f"{OUT}/selection.json"))
if "selection_c" not in sel:
    raise RuntimeError(f"{EXPORT} on GitHub has no selection C yet: push the latest commits and run again")
t = sel["totals"]
print(f"{t['images']} images, {t['boxes']} boxes; {t['images_scored']} images with ground truth")
for arm, m in t["per_model"].items():
    print(f"  {arm}: fully correct {m['fully_correct']} ({100 * m['fully_correct_share']:.1f}%), "
          f"mean per-image F1 {m['mean_f1']:.3f}, TP {m['tp']}, FP {m['fp']}, FN {m['fn']}")
print(f"  fully correct for all three models: {t['fully_correct_all_three']}, for at least two: "
      f"{t['fully_correct_at_least_two']}, for none: {t['fully_correct_none']}")
print("  fully correct by objects per image:")
for k, g in t["fully_correct_by_objects"].items():
    print(f"    {k:>3} object(s): {g['images']} images; "
          + ", ".join(f"{arm} {g[arm]}" for arm in t["per_model"]) + f", all three {g['all_three']}")
for key in ("selection_a", "selection_b", "selection_c"):
    s = sel[key]
    print(f"\n{key}: {s['purpose']}\n  rule: {s['rule']}")
    if key == "selection_b":
        for lv in s["levels"]:
            print(f"  level {lv['level']}: n_gt >= {lv['min_gt']}, fully correct for >= "
                  f"{lv['min_models_fully_correct']} models: {lv['candidates']} candidates")
        print(f"  level used: {s['level_used']}")
        if len(s["images"]) < 3:
            print(f"  WARNING: only {len(s['images'])} best-case images even at the last level")
    if key == "selection_c":
        print(f"  candidates: {s['candidates']}")
        if len(s["images"]) < 2:
            print(f"  WARNING: only {len(s['images'])} single-object images are fully correct for all three models")
    for e in s["images"]:
        stats = ", ".join(f"{a} TP {m['tp']}/FP {m['fp']}/FN {m['fn']}" for a, m in e["models"].items())
        print(f"  {e['file']} [{e['source']}] n_gt {e['n_gt']}: {stats}")
print(f"\noverlaps: A-B {sel['overlap_a_b']}, A-C {sel['overlap_a_c']}, B-C {sel['overlap_b_c']}")
names = zipfile.ZipFile(ZIP).namelist()
print(f"\n{ZIP}: {os.path.getsize(ZIP) / 1e6:.2f} MB, {len(names)} files")
for n in names:
    print("  ", n)


### Files to download / where they go locally

From the **Output** tab of the finished version:

| File in the Output tab | Where it goes locally |
|---|---|
| `realworld_samples.zip` | unzip it into `paper/figures/data/realworld/` (so that `paper/figures/data/realworld/selection.json` exists) |

Then draw the figures locally (project venv): `.venv/Scripts/python.exe paper/figures/scripts/r_samples.py`. With the
bundle in place it adds the real-world row to `r_samples.jpg` and writes `r_best.jpg` (the best cases), and it prints
the rule level and the totals for the captions. Commit the bundle together with the two figures.

The `realworld_samples/` folder in the Output tab holds the same files unzipped; the `SwarmSort/` clone and the
`gen2_cfg/` configs are not needed.

### If the session dies

Run it again with the same input: the predictions and the selection rules are deterministic, so it gives the same
bundle, and nothing needs to be carried over.